# 5 · Time Series
*Data Visualization for Scientists & Public Health Professionals*

Data indexed by time has its own toolkit: you plot it as a line, you usually need to **smooth** it to see the signal under the noise, you have to decide what to do about **gaps**, and you often want to **annotate** the events that explain the bumps. This notebook covers all of that, using daily respiratory-illness surveillance data.

We use `resp_ed_visits`: daily counts of emergency-department visits for respiratory illness across 2023–2024, alongside `pct_positive`, the percent of respiratory tests coming back positive. Respiratory illness is strongly seasonal, so this is a dataset with a real signal to find — and a deliberate reporting gap to handle honestly.

### Learning objectives
- Confirm a date column is a real datetime and plot a series against it
- Resample to a coarser frequency to reveal the trend
- Smooth with a rolling average, and choose the window deliberately
- Put two related series in context together
- Handle gaps honestly
- Annotate an event on a time axis

### Agenda
1. Dates as dates
2. Resampling
3. Rolling averages
4. A second series in context
5. Gaps in the data
6. Annotating events

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## Setup

Everything here depends on the date column being a real **datetime**, not text. Parse it on load with `parse_dates=`; once it is a datetime, matplotlib spaces points by actual time and formats the axis for you.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
resp = pd.read_csv(f"{BASE_URL}/resp_ed_visits.csv", parse_dates=["date"])
print(resp["date"].dtype)
resp.head()

## 1. Dates as dates

The raw daily series shows the shape, but it is busy: day-to-day noise and a weekend dip ride on top of the slow seasonal wave. That is the situation smoothing is for — but first, plot it honestly so you can see what you are up against.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(resp["date"], resp["resp_ed_visits"], linewidth=0.7)
ax.set_title("Respiratory ED visits, daily (2023–2024)")
ax.set_xlabel("Date")
ax.set_ylabel("ED visits")
fig.tight_layout()
plt.show()

> **Note:** The winter peaks and summer troughs are already visible, but so is a lot of daily jitter. Because `date` is a real datetime, the x-axis is spaced by actual time and labeled with years and months for free.

## 2. Resampling

**Resampling** changes the time frequency. With a datetime index, `resample("W").mean()` collapses daily data to a weekly average; `"ME"` gives month-end. This averages away the day-to-day noise and leaves the seasonal signal. The frequency aliases are the modern ones: `"D"` daily, `"W"` weekly, `"ME"` month-end, `"QE"` quarter-end, `"YE"` year-end.

In [ ]:
weekly = resp.set_index("date")["resp_ed_visits"].resample("W").mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weekly.index, weekly.values)
ax.set_title("Weekly mean respiratory ED visits")
ax.set_xlabel("Week")
ax.set_ylabel("Mean visits per day")
fig.tight_layout()
plt.show()

> **Note:** Resampling needs a **datetime index** — hence the `set_index("date")`. The seasonal wave that was buried in daily jitter is now unmistakable: two winter surges, two quiet summers. [pandas resampling guide](https://pandas.pydata.org/docs/user_guide/timeseries.html#resampling)

## 3. Rolling averages

A **rolling** (moving) average smooths differently: it keeps every day but replaces each value with the average of a window around it. A 7-day window spans exactly one week, so it cancels the weekend dip while preserving daily resolution. Resampling gives you fewer, coarser points; rolling keeps the same points but smoother.

In [ ]:
s = resp.set_index("date")["resp_ed_visits"]
rolling7 = s.rolling(7).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(s.index, s.values, linewidth=0.7, alpha=0.4, label="daily")
ax.plot(rolling7.index, rolling7.values, color="crimson", label="7-day average")
ax.set_title("Respiratory ED visits: daily vs. 7-day rolling mean")
ax.set_xlabel("Date")
ax.set_ylabel("ED visits")
ax.legend()
fig.tight_layout()
plt.show()

> **Note:** The first six days are blank because a 7-day window cannot fill until it has seven values. The window width is an editorial choice — wider is smoother but slower to react. The cleanest way to feel that trade-off is to move it yourself.

### Feel the smoothing window

Run the cell and drag the window from 1 day (raw, spiky) up to 60 (very smooth, but slow to turn at the peaks). Notice the trade: a wide window kills noise but also flattens and delays the real winter surge. (The slider needs a live kernel.)

In [ ]:
from ipywidgets import interact, IntSlider

def plot_rolling(window=7):
    s = resp.set_index("date")["resp_ed_visits"]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(s.index, s.values, linewidth=0.6, alpha=0.3, label="daily")
    ax.plot(s.index, s.rolling(window).mean(), color="crimson", label=f"{window}-day average")
    ax.set_title(f"Rolling window = {window} days")
    ax.set_xlabel("Date")
    ax.set_ylabel("ED visits")
    ax.legend()
    fig.tight_layout()
    plt.show()

interact(plot_rolling, window=IntSlider(min=1, max=60, step=1, value=7));

### Exercise 1 — Resample or roll? *(6 min)*

Summarize the daily `resp_ed_visits` series two ways on one chart, over the faint daily line: a **monthly resample** (`"ME"`) and a **30-day rolling mean**. Label it. In a comment, decide which summary answers *"when is the season?"* and which answers *"how sharp was this year's peak?"*

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
s = resp.set_index("date")["resp_ed_visits"]
monthly = s.resample("ME").mean()
rolling30 = s.rolling(30).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(s.index, s.values, linewidth=0.6, alpha=0.3, label="daily")
ax.plot(rolling30.index, rolling30.values, color="crimson", label="30-day rolling")
ax.plot(monthly.index, monthly.values, marker="o", color="black", label="monthly resample")
ax.set_title("Same signal: rolling mean vs. monthly resample")
ax.set_xlabel("Date")
ax.set_ylabel("ED visits")
ax.legend()
fig.tight_layout()
plt.show()
# The monthly resample gives few clean points -- best for "when is the season?". The
# 30-day rolling keeps daily resolution -- best for "how fast did this peak climb?". Same
# data, two summaries: which fits the question?
```

**Why this works.** Both summaries ride over the same faint daily line, so the contrast is clean. `resample("ME").mean()` collapses the year to twelve points — coarse and uncluttered, ideal for locating the season. `rolling(30).mean()` keeps every day but smooths the noise, preserving how steeply a peak climbs. Same series, two questions, two right tools.

</details>

## 4. A second series in context

Surveillance rarely rests on one number. `pct_positive` — the share of respiratory tests coming back positive — tells you whether rising visits reflect more actual disease or just more testing. The two live on different scales (counts vs. percent), so the honest way to compare them is **stacked panels sharing an x-axis**, not two lines forced onto one.

In [ ]:
wk_visits = resp.set_index("date")["resp_ed_visits"].resample("W").mean()
wk_pct = resp.set_index("date")["pct_positive"].resample("W").mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

ax1.plot(wk_visits.index, wk_visits.values, color="steelblue")
ax1.set_title("Respiratory ED visits and test positivity move together")
ax1.set_ylabel("Mean visits per day")

ax2.plot(wk_pct.index, wk_pct.values, color="seagreen")
ax2.set_xlabel("Week")
ax2.set_ylabel("Percent positive")

fig.tight_layout()
plt.show()

> **Note:** Sharing the x-axis lines the two stories up in time: the visit surges and the positivity surges rise and fall together, which is the reassuring sign that the visit spikes are real disease, not a testing artifact. Two different-scale series *can* be drawn on one plot with a second y-axis (`ax.twinx()`), but that trick is easy to make misleading, so reach for stacked panels first.

## 5. Gaps in the data

The positivity feed went dark for eleven days in March 2024 — a lab reporting outage — so those rows are missing (`NaN`). matplotlib's honest default is to **break the line** at a gap rather than draw through it. Zoom in on that window to see the break, and compare it to a version that fills the gap by interpolation.

In [ ]:
window = resp[(resp["date"] >= "2024-02-15") & (resp["date"] <= "2024-04-10")]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].plot(window["date"], window["pct_positive"].interpolate(), marker=".", color="crimson")
axes[0].set_title("Misleading: interpolated straight through")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Percent positive")
axes[0].tick_params(axis="x", rotation=30)

axes[1].plot(window["date"], window["pct_positive"], marker=".")
axes[1].set_title("Honest: the line breaks at the gap")
axes[1].set_xlabel("Date")
axes[1].tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

> **Warning:** The break is a feature: it tells the reader data is *missing*, not that positivity fell. The interpolated version invents a smooth run of values that were never measured. If you must connect across a gap, `series.interpolate()` will do it — but label it as an estimate, because it is one, and never let a fill quietly pass as observed data.

### Exercise 2 — Show the gap honestly *(7 min)*

Plot daily `pct_positive` for all of **2024**, fully labeled, and leave the gap unfilled. In a comment, say what the break in the line tells a reader — and what would be lost if you interpolated across it.

In [ ]:
# Your work here


<details>
<summary><b>Solution</b></summary>

```python
y2024 = resp[resp["date"].dt.year == 2024]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y2024["date"], y2024["pct_positive"], linewidth=1)
ax.set_title("Respiratory test positivity, 2024")
ax.set_xlabel("Date")
ax.set_ylabel("Percent positive")
fig.tight_layout()
plt.show()
# The break in March marks eleven days with no reporting. Interpolating would hide the
# outage and imply we measured positivity we never actually observed.
```

**Why this works.** Plotting the series with its `NaN` values left in place lets matplotlib do the honest thing automatically — it simply lifts the pen over the missing dates. The visible gap is information: it documents a data-collection problem the reader deserves to know about.

</details>

## 6. Annotating events

A time series usually needs context: *why* is there a peak there? Mark it. `ax.axvspan` shades a date range, `ax.axvline` drops a rule at a single date, and `ax.annotate` adds a label. Here we shade the first winter surge and label it, on the smoothed series so the annotation sits on a clean line.

In [ ]:
rolling7 = resp.set_index("date")["resp_ed_visits"].rolling(7).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(rolling7.index, rolling7.values, color="steelblue")

start, end = pd.Timestamp("2023-12-01"), pd.Timestamp("2024-02-15")
ax.axvspan(start, end, color="crimson", alpha=0.12)
ax.annotate("Winter surge", xy=(start, ax.get_ylim()[1]),
            xytext=(8, -14), textcoords="offset points", color="crimson")

ax.set_title("Respiratory ED visits (7-day mean) with the winter surge marked")
ax.set_xlabel("Date")
ax.set_ylabel("ED visits")
fig.tight_layout()
plt.show()

> **Note:** A shaded span plus a short label turns an unexplained peak into a documented event. Annotation is where a chart stops being exploratory and becomes a finished piece of communication — the focus of the final notebook.

## Wrap-up

You can now confirm a date is a real datetime and plot against it, resample to a coarser frequency to expose the trend, smooth with a rolling average whose window you choose deliberately, put two related series in honest context, handle gaps without deceiving anyone, and annotate the events that explain the data.

**Next:** Faceting and polishing — small multiples, accessible color, themes, and taking a chart all the way to presentation-ready — closing with the second half of the capstone.